In [1]:
%pip install datasets ipywidgets gensim wefe

Note: you may need to restart the kernel to use updated packages.


In [15]:
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec
from datasets import load_dataset
import os
import gensim.downloader as api
from gensim.models.fasttext import load_facebook_model
from wefe.datasets import load_weat
from wefe.metrics import WEAT
from wefe.query import Query
from wefe.word_embedding_model import WordEmbeddingModel

google_model = api.load("word2vec-google-news-300")
glove_model = api.load("glove-wiki-gigaword-100")

In [3]:
# Using a recent stable snapshot
dataset = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train")
# check the first example:
print(dataset[0]["text"][:200])

April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.

April always begins on the same day 


In [4]:
print(type(dataset))
print(dataset[0])

<class 'datasets.arrow_dataset.Dataset'>
{'id': '1', 'url': 'https://simple.wikipedia.org/wiki/April', 'title': 'April', 'text': 'April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.\n\nApril always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.\n\nThe Month \n\nApril comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.\n\nApril begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other\'s last days are exactly 35 weeks (245 days) apart.\n\nIn common years, April starts on the same day of the week as October of t

In [5]:
# This was a suggestion from ChatGPT and I love it

def tokenize(example):
    return {"tokens": simple_preprocess(example["text"])}

dataset = dataset.map(tokenize)


In [6]:
print(dataset[0])

{'id': '1', 'url': 'https://simple.wikipedia.org/wiki/April', 'title': 'April', 'text': 'April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.\n\nApril always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.\n\nThe Month \n\nApril comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.\n\nApril begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other\'s last days are exactly 35 weeks (245 days) apart.\n\nIn common years, April starts on the same day of the week as October of the previous year, and in leap years, May 

In [7]:
sentences = dataset["tokens"]
print(sentences[0])

['april', 'apr', 'is', 'the', 'fourth', 'month', 'of', 'the', 'year', 'in', 'the', 'julian', 'and', 'gregorian', 'calendars', 'and', 'comes', 'between', 'march', 'and', 'may', 'it', 'is', 'one', 'of', 'the', 'four', 'months', 'to', 'have', 'days', 'april', 'always', 'begins', 'on', 'the', 'same', 'day', 'of', 'the', 'week', 'as', 'july', 'and', 'additionally', 'january', 'in', 'leap', 'years', 'april', 'always', 'ends', 'on', 'the', 'same', 'day', 'of', 'the', 'week', 'as', 'december', 'the', 'month', 'april', 'comes', 'between', 'march', 'and', 'may', 'making', 'it', 'the', 'fourth', 'month', 'of', 'the', 'year', 'it', 'also', 'comes', 'first', 'in', 'the', 'year', 'out', 'of', 'the', 'four', 'months', 'that', 'have', 'days', 'as', 'june', 'september', 'and', 'november', 'are', 'later', 'in', 'the', 'year', 'april', 'begins', 'on', 'the', 'same', 'day', 'of', 'the', 'week', 'as', 'july', 'every', 'year', 'and', 'on', 'the', 'same', 'day', 'of', 'the', 'week', 'as', 'january', 'in', 'l

In [8]:
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=200, # I seem to have the machine for this, so why not?   
    window=7, # Per ChatGPT, Small (2–5): more syntactic relationships, Large (5–10): more semantic relationships          
    min_count=5,
    workers=os.cpu_count(),
    sg=0               
)

cbow_model.save("simplewiki_word2vec_cbow.model")

skip_g_model = Word2Vec(
    sentences=sentences,
    vector_size=200,   
    window=7,          
    min_count=1,       
    workers=os.cpu_count(),
    sg=1               
)
skip_g_model.save("simplewiki_word2vec_skip_g.model")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [9]:
print(cbow_model.wv.most_similar(positive=["king", "woman"], negative=["man"]))
print(skip_g_model.wv.most_similar(positive=["king", "woman"], negative=["man"]))
 

[('queen', 0.636311411857605), ('consort', 0.5963636636734009), ('empress', 0.5229761600494385), ('monarch', 0.5159528255462646), ('throne', 0.5083964467048645), ('regent', 0.5017926096916199), ('princess', 0.4993875026702881), ('ruler', 0.4816319942474365), ('countess', 0.46584463119506836), ('wangchuck', 0.45970839262008667)]
[('queen', 0.6789166331291199), ('consort', 0.629179060459137), ('margrethe', 0.5858632922172546), ('princess', 0.5752996802330017), ('hellenes', 0.5667928457260132), ('meresankh', 0.5665040016174316), ('regnant', 0.5654832124710083), ('huni', 0.5597191452980042), ('khesar', 0.5595388412475586), ('namgyel', 0.5592958927154541)]


In [10]:
print(cbow_model.wv.most_similar("piano", topn=10))
print(skip_g_model.wv.most_similar("piano", topn=10))

[('violin', 0.8129768371582031), ('clarinet', 0.8048903942108154), ('harpsichord', 0.7957672476768494), ('cello', 0.7950131297111511), ('concerto', 0.7465722560882568), ('pianos', 0.7335599064826965), ('violins', 0.7232279181480408), ('flute', 0.721782386302948), ('concertos', 0.7093640565872192), ('oboe', 0.6984553337097168)]
[('violin', 0.8084601163864136), ('cello', 0.7999385595321655), ('harpsichord', 0.7845757603645325), ('oboe', 0.7445868849754333), ('sonatas', 0.7425000071525574), ('mandolin', 0.7424894571304321), ('clarinet', 0.737724244594574), ('concerto', 0.737264096736908), ('violins', 0.7350459694862366), ('saxophones', 0.7326149940490723)]


In [11]:
def run_query(model_name, model, sim_words, ant_words=None, top_n=None):
    if hasattr(model, 'wv'):  # full Word2Vec object
        wv = model.wv
    else:  # pre-trained KeyedVectors
        wv = model
    try:
        if top_n:
            if isinstance(sim_words, str):
                sim_words = [sim_words]
            query = wv.most_similar(sim_words, topn=top_n)
        else:
            query = wv.most_similar(positive=sim_words, negative=ant_words)
    except KeyError:
        query = ['No answer']
    print_answer(model_name, sim_words, query, ant_words, top_n) 
    return query

def print_answer(model_name, sim_words, query, ant_words=None, top_n=None):
    if ant_words:
        print(f'Model: {model_name} {sim_words} - {ant_words}: ')
    else:
        print(f'Model: {model_name} {sim_words}: ')
    print([word[0] for word in query])
    print(f'Highest Probability: {query[0][1]}')
    print(f'Lowest Probability: {query[-1][1]}\n')


q1_cbow = run_query('CBOW', cbow_model, ['month', 'april'], ant_words=['year'], top_n=10)

Model: CBOW ['month', 'april'] - ['year']: 
['march', 'june', 'february', 'july', 'january', 'december', 'november', 'august', 'october', 'september']
Highest Probability: 0.7524385452270508
Lowest Probability: 0.6239780187606812



In [12]:
# Queries

# Top 10s
top_n = 10
queries = {'q1':{'sim_words':['month', 'april'], 'ant_words': ['year']},
                  'q2':{'sim_words':['king', 'woman'], 'ant_words': ['man']},
                  'q3':{'sim_words':['country', 'music'], 'ant_words': ['argentina']},
                  'q4':{'sim_words':['music', 'instrument'], 'ant_words': ['rnb']},
                  'q5':{'sim_words':['mswati', 'king'], 'ant_words': ['africa']},
                 }               
word_models = {'CBOW':cbow_model, 'Skip_G':skip_g_model, 'Google':google_model, 'GLoVE':glove_model}

for word_model in word_models:
    for query in queries:
        run_query(word_model, word_models[word_model],queries[query]['sim_words'][0], top_n)
        run_query(word_model, word_models[word_model], queries[query]['sim_words'], queries[query]['ant_words'])


Model: CBOW month - 10: 
['year', 'kōken', 'shōhei', 'một', 'kimmei', 'tenshō', 'tenures', 'kōei', 'hōjō', 'kōkoku']
Highest Probability: 0.4417212903499603
Lowest Probability: 0.34805238246917725

Model: CBOW ['month', 'april'] - ['year']: 
['march', 'february', 'july', 'december', 'june', 'august', 'january', 'november', 'october', 'september']
Highest Probability: 0.7425194978713989
Lowest Probability: 0.6382865309715271

Model: CBOW king - 10: 
['ruler', 'consort', 'khesar', 'descendant', 'wangchuck', 'namgyel', 'confessor', 'kings', 'vassal', 'royalty']
Highest Probability: 0.5067077279090881
Lowest Probability: 0.46078404784202576

Model: CBOW ['king', 'woman'] - ['man']: 
['queen', 'consort', 'empress', 'monarch', 'throne', 'regent', 'princess', 'ruler', 'countess', 'wangchuck']
Highest Probability: 0.636311411857605
Lowest Probability: 0.45970839262008667

Model: CBOW country - 10: 
['nation', 'republics', 'oppressed', 'continent', 'pelota', 'poorest', 'ethnic', 'influence', 's

In [13]:
# This query came from the WEAT docs but it servess the purpose

gender_query = Query(
    target_sets=[
        ["female", "woman", "girl", "sister", "she", "her", "hers", "daughter"],
        ["male", "man", "boy", "brother", "he", "him", "his", "son"],
    ],
    attribute_sets=[
        [
        "home",
        "parents",
        "children",
        "family",
        "cousins",
        "marriage",
        "wedding",
        "relatives",
        ],
        [
        "executive",
        "management",
        "professional",
        "corporation",
        "salary",
        "office",
        "business",
        "career",
        ],
    ],
    target_sets_names=["Female terms", "Male Terms"],
    attribute_sets_names=["Family", "Careers"],
)


In [21]:

for model in word_models:   
    if hasattr(word_models[model], 'wv'):  
        wv = word_models[model].wv
    else:
        wv = word_models[model]
    wem = WordEmbeddingModel(wv)
    metric = WEAT()
    result = metric.run_query(gender_query, wem)
    print(model)
    print(result)

CBOW
{'query_name': 'Female terms and Male Terms wrt Family and Careers', 'result': 0.6683310544467531, 'weat': 0.6683310544467531, 'effect_size': 0.7639260722669375, 'p_value': nan}
Skip_G
{'query_name': 'Female terms and Male Terms wrt Family and Careers', 'result': 0.4478179467841983, 'weat': 0.4478179467841983, 'effect_size': 0.577195968778509, 'p_value': nan}
Google
{'query_name': 'Female terms and Male Terms wrt Family and Careers', 'result': 0.4634387474798132, 'weat': 0.4634387474798132, 'effect_size': 0.45076521715906076, 'p_value': nan}
GLoVE
{'query_name': 'Female terms and Male Terms wrt Family and Careers', 'result': 0.8199977553449571, 'weat': 0.8199977553449571, 'effect_size': 0.9539907417802495, 'p_value': nan}


In [17]:
! wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
! tar -xzf aclImdb_v1.tar.gz

--2026-03-18 19:02:21--  https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘aclImdb_v1.tar.gz.1’

aclImdb_v1.tar.gz.1 100%[===================>]  80.23M  12.0MB/s    in 9.6s    

2026-03-18 19:02:31 (8.35 MB/s) - ‘aclImdb_v1.tar.gz.1’ saved [84125825/84125825]



In [18]:

# Class Example of a Sentiment Classifier

import glob
pos_train_files = glob.glob('aclImdb/train/pos/*')
neg_train_files = glob.glob('aclImdb/train/neg/*')
print(pos_train_files[:5])

from sklearn.feature_extraction.text import TfidfVectorizer
# only use 1000 data points per class for now to make things faster/simpler
num_files_per_class = 1000
all_train_files = pos_train_files[:num_files_per_class] + neg_train_files[:num_files_per_class]
vectorizer = TfidfVectorizer(input="filename", stop_words="english")
vectors = vectorizer.fit_transform(all_train_files)
vectors

import random
import numpy as np
from scipy.special import expit

def sgd_for_lr_with_ce(X, y, num_passes=5, learning_rate = 0.1, cbow=False):

    num_data_points = X.shape[0]

    # Initialize theta -> 0
    num_features = X.shape[1]
    # set w and b to the correct values
    w = np.zeros(num_features)
    b = 0

    # repeat until done
    # how to define "done"? let's just make it num passes for now
    # we can also do norm of gradient and when it is < epsilon (something tiny)

    for current_pass in range(num_passes):

        # iterate through entire dataset in random order
        order = list(range(num_data_points))
        random.shuffle(order)
        for i in order:

            # compute y-hat for this value of i given y_i and x_i
            
            if cbow: # Added this flag so CBOW would work because I'm lazy
                x_i = X[i]
            
            else:
                x_i = X[i].todense()
            y_i = y[i]

            # need to compute y_hat based on w and b, and x
            # sigmoid(w dot x + b)
            # use expit for sigmoid

            y_hat_i = expit(np.dot(x_i,w.T) + b)

            # for each w (and b), modify by -lr * (y_hat_i - y_i) * x_i
            w = w - learning_rate * (y_hat_i - y_i) * x_i
            b = b - learning_rate * (y_hat_i - y_i)

    # return theta
    return w,b

X = vectors
y = [1] * num_files_per_class + [0] * num_files_per_class
len(y)
w,b = sgd_for_lr_with_ce(X,y)

sorted_vocab = sorted([(k,v) for k,v in vectorizer.vocabulary_.items()],key=lambda x:x[1])
sorted_vocab = [(a,b) for (a,b) in sorted_vocab]
sorted_words_weights = sorted([x for x in zip(sorted_vocab, w.tolist()[0])], reverse=True, key=lambda x:x[1])

from sklearn.metrics import classification_report
w,b = sgd_for_lr_with_ce(X, y, num_passes=10)

# get the predictions
def predict_y_lr(w,b,X,threshold=0.5):

    # use our matrix operation version of the logistic regression model
    # X dot w + b
    # need to make w a column vector so the dimensions line up correctly
    y_hat = X.dot( w.reshape((-1,1)) ) + b

    # then just check if it's > threshold
    preds = np.where(y_hat > threshold,1,0)

    return preds

preds = predict_y_lr(w,b,X)

y_pred = predict_y_lr(w,b,X)
print(classification_report(y, y_pred))
    

['aclImdb/train/pos/11329_9.txt', 'aclImdb/train/pos/4582_7.txt', 'aclImdb/train/pos/10121_8.txt', 'aclImdb/train/pos/10586_10.txt', 'aclImdb/train/pos/9131_9.txt']
              precision    recall  f1-score   support

           0       0.81      1.00      0.90      1000
           1       1.00      0.77      0.87      1000

    accuracy                           0.88      2000
   macro avg       0.91      0.88      0.88      2000
weighted avg       0.91      0.88      0.88      2000



In [19]:
# Discrete BOW

from sklearn.feature_extraction.text import CountVectorizer

dbow_vectorizer = CountVectorizer(input="filename", stop_words="english")
dbow_X = dbow_vectorizer.fit_transform(all_train_files)
dbow_w, dbow_b = sgd_for_lr_with_ce(dbow_X, y, num_passes=10)
dbow_preds = predict_y_lr(w,b,dbow_X)

dbow_y_pred = predict_y_lr(dbow_w,dbow_b,X)
print(classification_report(y, dbow_y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       1.00      1.00      1.00      1000

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [20]:
# CBOW Stuff

def average_embed(filename, model):
    with open(filename, 'r') as f:
        lines = f.read()
    words = lines.lower().split()
    vectors = []
    for word in words:
        if word in model:
            vectors.append(model[word])
    return np.mean(vectors, axis=0)

cbow_X_emb = []
for file in all_train_files:
    cbow_X_emb.append(average_embed(file, cbow_model.wv))
cbow_X_emb = np.array(cbow_X_emb)
cbow_w,cbow_b = sgd_for_lr_with_ce(cbow_X_emb, y, num_passes=10, cbow=True)

def predict_cbow(w, b, X, threshold=0.5):
    y_hat = X.dot(w) + b
    return np.where(y_hat > threshold, 1, 0)

cbow_y_pred = predict_cbow(cbow_w, cbow_b, cbow_X_emb)

print(classification_report(y, cbow_y_pred))


              precision    recall  f1-score   support

           0       0.93      0.25      0.40      1000
           1       0.57      0.98      0.72      1000

    accuracy                           0.62      2000
   macro avg       0.75      0.62      0.56      2000
weighted avg       0.75      0.62      0.56      2000

